In [32]:
import numpy as np
import pandas as pd

In [33]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 5000

data = {
    "customer_id": [f"CUST{i:05d}" for i in range(1, n + 1)],
    "gender": np.random.choice(["Male", "Female"], n),
    "age": np.random.randint(18, 71, n),
    "tenure_months": np.random.randint(1, 73, n),
    "contract_type": np.random.choice(
        ["Monthly", "One Year", "Two Year"],
        n,
        p=[0.55, 0.25, 0.20]
    ),
    "monthly_charges": np.round(
        np.random.uniform(500, 5000, n), 2
    ),
    "payment_method": np.random.choice(
        ["Credit Card", "Debit Card", "UPI", "Bank Transfer", "Cash"],
        n
    ),
    "internet_service": np.random.choice(
        ["Fiber", "DSL", "None"],
        n,
        p=[0.50, 0.35, 0.15]
    ),
    "support_tickets": np.random.poisson(2, n),
    "complaints": np.random.poisson(1, n),
    "avg_monthly_usage_gb": np.round(
        np.random.uniform(5, 500, n), 2
    ),
    "login_frequency": np.random.randint(1, 31, n),
    "discount_percent": np.random.choice(
        [0, 5, 10, 15, 20],
        n,
        p=[0.35, 0.25, 0.20, 0.15, 0.05]
    ),
    "satisfaction_score": np.random.randint(1, 11, n),
    "late_payments": np.random.poisson(1, n)
}

df = pd.DataFrame(data)

df.head()

,customer_id,gender,age,tenure_months,contract_type,monthly_charges,payment_method,internet_service,support_tickets,complaints,avg_monthly_usage_gb,login_frequency,discount_percent,satisfaction_score,late_payments
0,CUST00001,Male,68,26,Two Year,1315.66,Debit Card,None,2,2,310.00,30,20,1,1
1,CUST00002,Female,64,48,One Year,4660.91,Credit Card,Fiber,3,1,226.32,9,15,8,0
2,CUST00003,Male,64,19,Monthly,966.19,Debit Card,Fiber,1,2,67.11,13,15,6,0
3,CUST00004,Male,21,9,Monthly,2943.45,UPI,Fiber,1,0,408.99,8,5,4,2
4,CUST00005,Male,24,64,One Year,1913.04,Bank Transfer,Fiber,4,2,147.09,17,15,6,1


In [34]:
df["total_charges"] = np.round(
    df["monthly_charges"] * df["tenure_months"] * (1 - df["discount_percent"] / 100), 2
)

df.head()

,customer_id,gender,age,tenure_months,contract_type,monthly_charges,payment_method,internet_service,support_tickets,complaints,avg_monthly_usage_gb,login_frequency,discount_percent,satisfaction_score,late_payments,total_charges
0,CUST00001,Male,68,26,Two Year,1315.66,Debit Card,None,2,2,310.00,30,20,1,1,27365.73
1,CUST00002,Female,64,48,One Year,4660.91,Credit Card,Fiber,3,1,226.32,9,15,8,0,190165.13
2,CUST00003,Male,64,19,Monthly,966.19,Debit Card,Fiber,1,2,67.11,13,15,6,0,15603.97
3,CUST00004,Male,21,9,Monthly,2943.45,UPI,Fiber,1,0,408.99,8,5,4,2,25166.50
4,CUST00005,Male,24,64,One Year,1913.04,Bank Transfer,Fiber,4,2,147.09,17,15,6,1,104069.38


In [35]:
churn_score = (
    -0.8
    + 0.9 * (df["contract_type"] == "Monthly")
    - 0.025 * df["tenure_months"]
    + 0.12 * df["support_tickets"]
    + 0.18 * df["complaints"]
    + 0.20 * df["late_payments"]
    - 0.25 * df["satisfaction_score"]
    + 0.00015 * df["monthly_charges"]
)

churn_probability = 1 / (1 + np.exp(-churn_score))
df["churn"] = np.random.binomial(1, churn_probability)

df.head()

,customer_id,gender,age,tenure_months,contract_type,monthly_charges,payment_method,internet_service,support_tickets,complaints,avg_monthly_usage_gb,login_frequency,discount_percent,satisfaction_score,late_payments,total_charges,churn
0,CUST00001,Male,68,26,Two Year,1315.66,Debit Card,None,2,2,310.00,30,20,1,1,27365.73,0
1,CUST00002,Female,64,48,One Year,4660.91,Credit Card,Fiber,3,1,226.32,9,15,8,0,190165.13,0
2,CUST00003,Male,64,19,Monthly,966.19,Debit Card,Fiber,1,2,67.11,13,15,6,0,15603.97,0
3,CUST00004,Male,21,9,Monthly,2943.45,UPI,Fiber,1,0,408.99,8,5,4,2,25166.50,0
4,CUST00005,Male,24,64,One Year,1913.04,Bank Transfer,Fiber,4,2,147.09,17,15,6,1,104069.38,0


In [36]:
df["churn"].value_counts()

churn
0    3913
1    1087
Name: count, dtype: int64

In [37]:
df["churn"].value_counts(normalize=True) * 100

churn
0    78.26
1    21.74
Name: proportion, dtype: float64

In [38]:
missing_columns = [
    "monthly_charges",
    "total_charges",
    "internet_service",
    "satifaction_score",
    "payment_method"
]

for column in missing_columns:
    missing_indices = np.random.choice(df.index, size=int(0.02 * n), replace=False)
    df.loc[missing_indices, column] = np.nan

In [39]:
df.isnull().sum()

customer_id                0
gender                     0
age                        0
tenure_months              0
contract_type              0
monthly_charges          100
payment_method           100
internet_service         100
support_tickets            0
complaints                 0
avg_monthly_usage_gb       0
login_frequency            0
discount_percent           0
satisfaction_score         0
late_payments              0
total_charges            100
churn                      0
satifaction_score       5000
dtype: int64

In [40]:
import os

os.makedirs("Data/Raw", exist_ok=True)

df.to_csv("Data/Raw/customer_churn_data.csv", index=False)

print("File saved successfully!")

File saved successfully!


In [41]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customer_id           5000 non-null   str    
 1   gender                5000 non-null   str    
 2   age                   5000 non-null   int32  
 3   tenure_months         5000 non-null   int32  
 4   contract_type         5000 non-null   str    
 5   monthly_charges       4900 non-null   float64
 6   payment_method        4900 non-null   str    
 7   internet_service      4900 non-null   str    
 8   support_tickets       5000 non-null   int32  
 9   complaints            5000 non-null   int32  
 10  avg_monthly_usage_gb  5000 non-null   float64
 11  login_frequency       5000 non-null   int32  
 12  discount_percent      5000 non-null   int64  
 13  satisfaction_score    5000 non-null   int32  
 14  late_payments         5000 non-null   int32  
 15  total_charges         4900 non-n

In [42]:
df["churn"].value_counts(normalize=True) * 100

churn
0    78.26
1    21.74
Name: proportion, dtype: float64